# Layer-1 head isolation interventions

For each layer-1 head $i$, this notebook keeps head $i$ intact and mean-ablates the other layer-1 heads. It reports newline precision and recall, then compares the resulting activations before layer 2 attention (`blocks.2.hook_resid_pre`) with the clean activations using MSE and the variance captured by an independently fitted 3D PCA.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import torch
from IPython.display import display
from sklearn.decomposition import PCA

PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

import project_main.checkpoints as project_checkpoints
import project_main.data as project_data
import project_main.model as project_model
import project_main.tokens as project_tokens

RUN_DIR = PROJECT_ROOT / "results/line_breaks_results"
CHECKPOINT_PATH = RUN_DIR / "checkpoints/final.pt"
LAYER = 1
EVAL_SEED = 28_001
CALIBRATION_SEED = 28_002
N_EVAL_SEQUENCES = 500
N_CALIBRATION_SEQUENCES = 1_024
BATCH_SIZE = 64
PCA_SEED = 28_003


In [ ]:
with open(RUN_DIR / "config.json", "r") as f:
    cfg = json.load(f)
device = "cuda" if torch.cuda.is_available() else "cpu"
vocab = project_tokens.build_vocab(cfg["task"])
model = project_model.build_model(cfg=cfg, device=device)
checkpoint = project_checkpoints.load_checkpoint(path=CHECKPOINT_PATH, model=model, map_location=device)
model.eval()
n_layers = cfg["model"]["n_layers"]
n_heads = cfg["model"]["n_heads"]
d_head = cfg["model"]["d_head"]
seq_len = cfg["task"]["seq_len"]
Z_SITE = f"blocks.{LAYER}.attn.hook_z"
LAYER2_PRE_SITE = f"blocks.{LAYER + 1}.hook_resid_pre"
print(f"Loaded checkpoint step {checkpoint['step']} on {device}")
print(f"Layer {LAYER}: {n_heads} heads; measuring {LAYER2_PRE_SITE}")


## Estimate mean replacement vectors

In [ ]:
@torch.no_grad()
def estimate_head_means():
    sums = torch.zeros(n_heads, d_head, dtype=torch.float64)
    n_vectors = 0
    for start in range(0, N_CALIBRATION_SEQUENCES, BATCH_SIZE):
        size = min(BATCH_SIZE, N_CALIBRATION_SEQUENCES - start)
        batch = project_data.make_batch(batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device, seed=CALIBRATION_SEED + start)
        _, cache = model.run_with_cache(batch.tokens, prepend_bos=False, names_filter=[Z_SITE])
        sums += cache[Z_SITE].detach().cpu().double().sum(dim=(0, 1))
        n_vectors += size * seq_len
    return (sums / n_vectors).float()

replacement_head_outputs = estimate_head_means()
print("Replacement norms:", replacement_head_outputs.norm(dim=-1).numpy())


## Run clean and head-isolation evaluations

In [ ]:
@torch.no_grad()
def run_condition(keep_head=None, replacement_outputs=None):
    if replacement_outputs is None:
        replacement_outputs = replacement_head_outputs
    activation_batches = []
    true_positives = actual_newlines = predicted_newlines = 0
    total_positions = 0
    for start in range(0, N_EVAL_SEQUENCES, BATCH_SIZE):
        size = min(BATCH_SIZE, N_EVAL_SEQUENCES - start)
        batch = project_data.make_batch(batch_size=size, vocab=vocab, task_cfg=cfg["task"], device=device, seed=EVAL_SEED + start)
        if keep_head is None:
            logits, cache = model.run_with_cache(batch.tokens, prepend_bos=False, names_filter=[LAYER2_PRE_SITE])
        else:
            def isolate_head(z, hook, keep_head=keep_head):
                z_replaced = z.clone()
                for head in range(n_heads):
                    if head != keep_head:
                        z_replaced[:, :, head, :] = replacement_outputs[head].to(device=z.device, dtype=z.dtype)
                return z_replaced
            captured = {}
            def capture_layer2_pre(activation, hook):
                captured["activation"] = activation.detach()
                return activation
            logits = model.run_with_hooks(
                batch.tokens, prepend_bos=False,
                fwd_hooks=[(Z_SITE, isolate_head), (LAYER2_PRE_SITE, capture_layer2_pre)],
            )
            activation_batches.append(captured["activation"].cpu().numpy())
        if keep_head is None:
            activation_batches.append(cache[LAYER2_PRE_SITE].detach().cpu().numpy())
        predictions = logits.argmax(dim=-1)
        target_newline = batch.targets == vocab.newline_id
        prediction_newline = predictions == vocab.newline_id
        true_positives += int((target_newline & prediction_newline).sum().item())
        actual_newlines += int(target_newline.sum().item())
        predicted_newlines += int(prediction_newline.sum().item())
        total_positions += predictions.numel()
    return {
        "activations": np.concatenate(activation_batches, axis=0),
        "newline_precision": true_positives / predicted_newlines if predicted_newlines else np.nan,
        "newline_recall": true_positives / actual_newlines if actual_newlines else np.nan,
        "newline_prediction_frequency": predicted_newlines / total_positions,
    }

conditions = {"clean": run_condition()}
for keep_head in range(n_heads):
    conditions[f"keep head {keep_head}"] = run_condition(keep_head=keep_head)

clean_flat = conditions["clean"]["activations"].reshape(-1, conditions["clean"]["activations"].shape[-1])
results_rows = []
for name, condition in conditions.items():
    flat = condition["activations"].reshape(-1, condition["activations"].shape[-1])
    mse = float(np.mean((flat - clean_flat) ** 2))
    condition_pca = PCA(n_components=3, svd_solver="randomized", random_state=PCA_SEED).fit(flat)
    results_rows.append({
        "condition": name,
        "newline_precision": condition["newline_precision"],
        "newline_recall": condition["newline_recall"],
        "newline_prediction_frequency": condition["newline_prediction_frequency"],
        "layer2_pre_activation_MSE_vs_clean": mse,
        "layer2_pre_activation_PCA3_variance_explained": float(condition_pca.explained_variance_ratio_.sum()),
    })
results = pd.DataFrame(results_rows)
display(results.style.format({
    "newline_precision": "{:.4f}", "newline_recall": "{:.4f}",
    "newline_prediction_frequency": "{:.4f}",
    "layer2_pre_activation_MSE_vs_clean": "{:.6f}",
    "layer2_pre_activation_PCA3_variance_explained": "{:.2%}",
}))


## Visual comparison

In [ ]:
plot_metrics = results.melt(
    id_vars="condition",
    value_vars=["newline_precision", "newline_recall"],
    var_name="metric", value_name="value",
)
fig = px.bar(plot_metrics, x="condition", y="value", color="metric", barmode="group", title="Newline precision and recall", range_y=[0, 1])
display(fig)

activation_plot = results.melt(
    id_vars="condition",
    value_vars=["layer2_pre_activation_MSE_vs_clean", "layer2_pre_activation_PCA3_variance_explained"],
    var_name="metric", value_name="value",
)
fig = px.bar(activation_plot, x="condition", y="value", color="metric", barmode="group", title="Layer-2 pre-attention activation comparison")
display(fig)


## Zero-ablation head-isolation results

In [ ]:
zero_replacement_outputs = torch.zeros_like(replacement_head_outputs)
zero_conditions = {"clean": conditions["clean"]}
for keep_head in range(n_heads):
    zero_conditions[f"keep head {keep_head}"] = run_condition(keep_head=keep_head, replacement_outputs=zero_replacement_outputs)
zero_results_rows = []
for name, condition in zero_conditions.items():
    flat = condition["activations"].reshape(-1, condition["activations"].shape[-1])
    condition_pca = PCA(n_components=3, svd_solver="randomized", random_state=PCA_SEED).fit(flat)
    zero_results_rows.append({
        "condition": name,
        "newline_precision": condition["newline_precision"],
        "newline_recall": condition["newline_recall"],
        "newline_prediction_frequency": condition["newline_prediction_frequency"],
        "layer2_pre_activation_MSE_vs_clean": float(np.mean((flat - clean_flat) ** 2)),
        "layer2_pre_activation_PCA3_variance_explained": float(condition_pca.explained_variance_ratio_.sum()),
    })
zero_results = pd.DataFrame(zero_results_rows)
display(zero_results.style.format({
    "newline_precision": "{:.4f}", "newline_recall": "{:.4f}",
    "newline_prediction_frequency": "{:.4f}",
    "layer2_pre_activation_MSE_vs_clean": "{:.6f}",
    "layer2_pre_activation_PCA3_variance_explained": "{:.2%}",
}))
